# EuroSAT-MS Data Exploration

## 1. Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from pathlib import Path

## 2. Locate dataset and select one sample image

In [ ]:
data_dir = Path("../data/EuroSAT_MS/Forest")  
tif_files = list(data_dir.rglob("*.tif"))

print(f"Number of .tif files found: {len(tif_files)}")

sample_path = tif_files[0]
print("Sample file:", sample_path)

## 3. Read the multispectral GeoTIFF image

In [ ]:
with rasterio.open(sample_path) as src:
    img = src.read()
    profile = src.profile

print("Image shape:", img.shape)
print("Data type:", img.dtype)
print("Raster profile:")
print(profile)

## 4. Extract the class label

In [ ]:
label = sample_path.parent.name
print("Label:", label)

## 5. Inspect per-band statistics

In [ ]:
print("Per-band statistics:")
for b in range(img.shape[0]):
    band = img[b]
    print(
        f"Band {b+1:02d} | "
        f"min={band.min()}, "
        f"max={band.max()}, "
        f"mean={band.mean():.2f}, "
        f"std={band.std():.2f}"
    )

## 6. Normalization function for visualization

In [ ]:
def normalize_band(x, lower=2, upper=98):
    x = x.astype("float32")
    lo, hi = np.percentile(x, (lower, upper))
    x = np.clip((x - lo) / (hi - lo + 1e-8), 0, 1)
    return x

## 7. Visualize an RGB composite image

In [ ]:
r = normalize_band(img[3])   # B04 = RED
g = normalize_band(img[2])   # B03 = GREEN
b = normalize_band(img[1])   # B02 = BLUE

rgb = np.stack([r, g, b], axis=-1)

plt.figure(figsize=(6, 6))
plt.imshow(rgb)
plt.title(f"RGB Composite - {label}")
plt.axis("off")
plt.show()

## 8. Define Sentinel-2 band names

In [ ]:
band_names = [
    "B01", "B02", "B03", "B04", "B05", "B06", "B07",
    "B08", "B8A", "B09", "B10", "B11", "B12"
]

## 9. Visualize all 13 spectral bands

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
axes = axes.flatten()

for i in range(len(axes)):
    ax = axes[i]
    if i < img.shape[0]:
        ax.imshow(normalize_band(img[i]), cmap="gray")
        ax.set_title(band_names[i])
    ax.axis("off")

plt.suptitle(f"All 13 Spectral Bands - {label}", fontsize=14)
plt.tight_layout()
plt.show()

## Observations

- Each EuroSAT-MS sample is a 13-channel multispectral GeoTIFF image.
- The sample used here has shape `(13, 64, 64)` and data type `uint16`.
- RGB visualization is created using Sentinel-2 bands B04, B03, and B02.
- The RGB composite looks coarse because each image patch is only 64×64 pixels.
- Different spectral bands show different visual patterns, indicating that non-RGB bands may contain useful information beyond visible channels.

## 10. Plot pixel value distributions of selected bands

In [ ]:
plt.figure(figsize=(10, 5))

selected_bands = {
    "B02 (Blue)": img[1],
    "B03 (Green)": img[2],
    "B04 (Red)": img[3],
    "B08 (NIR)": img[7],
}

for name, band in selected_bands.items():
    plt.hist(band.ravel(), bins=50, alpha=0.5, label=name)

plt.title("Pixel Value Distribution of Selected Bands")
plt.xlabel("Pixel value")
plt.ylabel("Frequency")
plt.legend()
plt.show()